# Parity GPU run: Unsloth 2026.11.10

Step-by-step instructions are in `docs/gpu-run/GPU-RUN.md`. In short:

1. **Runtime > Change runtime type > T4 GPU > Save** (do this first).
2. Run each cell below in order with its ▶ button, and wait for each to finish.
3. Cell 4 asks for a file: pick `parity-src-<commit>.zip` from the kit folder.
4. Cell 6 downloads `parity-gpu-results-....zip`. Hand that zip back.

No logins or tokens are needed or used.

In [ ]:
# Cell 2: which models to run. Each is "org/name@<40-character commit>".
# The default is the calibration model. For the M5 target runs, list the target releases here,
# but only once their predictions are committed and the Unsloth adapter supports them.
MODELS = [
    "Qwen/Qwen3-0.6B@c1899de289a04d12100db370d81485cdf75e47ca",
]

In [ ]:
# Cell 3: is there a GPU?
import shutil
import subprocess
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU > Save, then run the cells again from cell 2.")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# Cell 4: upload the code (made by build_kit.py with git archive) and unpack it.
import hashlib
import pathlib
import re
import shutil
import zipfile
from google.colab import files

print("Click 'Choose files' and pick parity-src-<commit>.zip from the kit folder.")
uploaded = files.upload()
names = [n for n in uploaded if re.fullmatch(r"parity-src-[0-9a-f]{12}( \(\d+\))?\.zip", n)]
if len(names) != 1:
    raise RuntimeError(f"Upload exactly one parity-src-<commit>.zip (got {list(uploaded)}). Run this cell again.")
archive = pathlib.Path("/content") / names[0]
archive.write_bytes(uploaded[names[0]])
ARCHIVE_SHA256 = hashlib.sha256(archive.read_bytes()).hexdigest()
with zipfile.ZipFile(archive) as zf:
    COMMIT = zf.comment.decode("ascii", "replace")  # git archive stores the commit id here
    if not re.fullmatch(r"[0-9a-f]{40}", COMMIT) or not names[0].startswith(f"parity-src-{COMMIT[:12]}"):
        raise RuntimeError("This zip wasn't made by build_kit.py: its commit id is missing or doesn't match its name.")
    ROOT = pathlib.Path("/content/parity")
    if ROOT.exists():
        shutil.rmtree(ROOT)
    zf.extractall(ROOT)
print(f"Code from commit {COMMIT}")
print(f"Archive sha256  {ARCHIVE_SHA256}")

In [ ]:
# Cell 5: install Unsloth's pinned environment, check the GPU route, run the checks (about 20 to 40 minutes).
# Keep this browser tab open while it runs.
import subprocess
import sys
command = [
    sys.executable, str(ROOT / "docs" / "gpu-run" / "colab_kit.py"),
    "--models", ",".join(MODELS), "--commit", COMMIT, "--archive-sha256", ARCHIVE_SHA256,
    "--out", "/content/parity-gpu-out",
]
RESULT_ZIP = None
proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
    if line.startswith("RESULT_ZIP="):
        RESULT_ZIP = line.split("=", 1)[1].strip()
exit_code = proc.wait()
print(f"\nExit code {exit_code}. Run cell 6 to download the results zip, whether or not this succeeded.")

In [ ]:
# Cell 6: download the results zip.
from google.colab import files
if not RESULT_ZIP:
    raise RuntimeError("No results zip was written. Save the whole output of cell 5 as a text file and hand that back.")
print(f"Downloading {RESULT_ZIP}")
files.download(RESULT_ZIP)